# Pooling ablation
## RAGTruth $\rightarrow$ PublicHearingBR

This read-only notebook compares the frozen three-seed RAGTruth confirmatory campaigns using Gated Attention, Mean, and Max pooling. The intended scientific difference is only the aggregation of `projected [B,N,128]`: Gated Attention is trainable, while Mean and Max are non-parametric.

**Decision criterion: PublicHearingBR zero-shot AUPRC.** The target is imbalanced (501/4,235 positives), so AUPRC is the primary ranking criterion used here. AUROC and Brier score are reported as complementary discrimination and score-quality diagnostics.

## Reproducible configuration

Configuration precedence is `NOTEBOOK_CONFIG`, then `notebooks/configs/pooling_ablation.local.yaml`. Copy `pooling_ablation.example.yaml` to that local path to use different frozen campaign directories. Paths must remain repository-relative.

In [ ]:
import json
import os
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import yaml
from IPython.display import display


POOLINGS = ("gated_attention", "mean", "max")
LABELS = {
    "gated_attention": "Gated Attention",
    "mean": "Mean",
    "max": "Max",
}
COLORS = {
    "gated_attention": "#1f77b4",
    "mean": "#f4a261",
    "max": "#2a9d8f",
}


def find_repo_root(start: Path | None = None) -> Path:
    start = (start or Path.cwd()).resolve()
    for candidate in (start, *start.parents):
        if (candidate / "notebooks" / "configs" / "pooling_ablation.example.yaml").is_file():
            return candidate
    raise FileNotFoundError("Repository root with the pooling ablation configuration was not found.")


def load_notebook_config(root: Path) -> tuple[Path, dict[str, Path]]:
    requested = os.environ.get("NOTEBOOK_CONFIG")
    config_path = Path(requested) if requested else root / "notebooks" / "configs" / "pooling_ablation.local.yaml"
    if not config_path.is_absolute():
        config_path = root / config_path
    if not config_path.is_file():
        raise FileNotFoundError(
            "Local notebook configuration not found. Copy\n"
            "notebooks/configs/pooling_ablation.example.yaml\n"
            "to notebooks/configs/pooling_ablation.local.yaml and configure the campaign paths."
        )
    raw = yaml.safe_load(config_path.read_text(encoding="utf-8"))
    experiments = raw.get("experiments") if isinstance(raw, dict) else None
    if not isinstance(experiments, dict):
        raise ValueError("Notebook configuration requires an experiments mapping.")
    resolved: dict[str, Path] = {}
    for pooling in POOLINGS:
        entry = experiments.get(pooling)
        value = entry.get("path") if isinstance(entry, dict) else None
        relative_path = Path(value) if isinstance(value, str) else None
        if not relative_path or relative_path.is_absolute() or ".." in relative_path.parts:
            raise ValueError(f"{pooling} must define a repository-relative path.")
        resolved[pooling] = root / relative_path
    return config_path, resolved


def load_json(path: Path) -> dict:
    with path.open(encoding="utf-8") as handle:
        return json.load(handle)


def add_bar_labels(axis, bars, errors=None, fmt="{:.3f}", padding=0.012) -> None:
    errors = np.zeros(len(bars)) if errors is None else np.asarray(errors)
    for bar, error in zip(bars, errors):
        axis.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + error + padding, fmt.format(bar.get_height()), ha="center", va="bottom", fontsize=8)


ROOT = find_repo_root()
os.chdir(ROOT)
config_path, experiment_paths = load_notebook_config(ROOT)
config_label = config_path.relative_to(ROOT) if config_path.is_relative_to(ROOT) else config_path
print(f"Configuration: {config_label}")
plt.rcParams.update({"figure.dpi": 120, "axes.spines.top": False, "axes.spines.right": False})

## Frozen protocol parity

All three campaigns must use the same RAGTruth dataset, grouped split, seeds, model and optimization protocol. This cell verifies equality of the frozen experiment configuration after removing the operational run name and the pooling identity.

In [ ]:
campaigns = {}
per_seed_frames = []
protocol_rows = []

for pooling in POOLINGS:
    run_dir = experiment_paths[pooling]
    if not run_dir.is_dir():
        raise FileNotFoundError(f"Campaign directory not found: {run_dir.relative_to(ROOT)}")
    frozen = load_json(run_dir / "frozen_protocol.json")
    root_manifest = load_json(run_dir / "manifest.json")
    payload = frozen["payload"]
    experiment = payload["experiment_config"]
    campaigns[pooling] = {"dir": run_dir, "payload": payload, "experiment": experiment}
    protocol_rows.append({
        "pooling": LABELS[pooling],
        "run signature": frozen["signature"],
        "architecture": experiment["architecture"],
        "dataset signature": payload["dataset_signature"],
        "split signature": payload["split_signature"],
        "seeds": str(payload["model_seeds"]),
        "selection": f"{payload['selection_metric']} ({payload['selection_mode']})",
        "status": root_manifest["status"],
    })
    frame = pd.read_csv(run_dir / "aggregate" / "per_seed_metrics.csv")
    frame["pooling"] = pooling
    per_seed_frames.append(frame)

reference = campaigns["gated_attention"]["experiment"].copy()
for field in ("run_name", "architecture", "pooling_type"):
    reference.pop(field, None)
for pooling in ("mean", "max"):
    candidate = campaigns[pooling]["experiment"].copy()
    for field in ("run_name", "architecture", "pooling_type"):
        candidate.pop(field, None)
    assert candidate == reference, f"Non-pooling configuration differs for {pooling}."

metrics = pd.concat(per_seed_frames, ignore_index=True)
display(pd.DataFrame(protocol_rows))
print("Protocol check passed: frozen configurations are identical outside pooling identity and the operational run name.")

## Threshold-free ranking performance

Continuous-score metrics are shown as mean $\pm$ standard deviation across seeds. They do not depend on a decision threshold. AUPRC is primary for the imbalanced target; AUROC and Brier provide complementary views.

In [ ]:
threshold_free = metrics.loc[metrics["threshold_regime"].eq("threshold_free")].copy()
summary = (
    threshold_free.groupby(["dataset", "pooling"])[["AUPRC", "AUROC", "Brier"]]
    .agg(["mean", "std"])
)

figure, axes = plt.subplots(1, 2, figsize=(12.0, 4.0), sharey=True)
for axis, dataset, title in zip(axes, ("ragtruth_test", "publichearing_zero_shot"), ("RAGTruth test", "PublicHearingBR zero-shot")):
    x = np.arange(2)
    width = 0.24
    for index, pooling in enumerate(POOLINGS):
        values = [summary.loc[(dataset, pooling), (metric, "mean")] for metric in ("AUPRC", "AUROC")]
        errors = [summary.loc[(dataset, pooling), (metric, "std")] for metric in ("AUPRC", "AUROC")]
        bars = axis.bar(x + (index - 1) * width, values, width, yerr=errors, capsize=3, label=LABELS[pooling], color=COLORS[pooling])
        add_bar_labels(axis, bars, errors)
    axis.set_xticks(x, ("AUPRC", "AUROC"))
    axis.set_ylim(0, 1)
    axis.set_title(title, loc="left", weight="bold")
    axis.grid(axis="y", alpha=0.2)
axes[0].set_ylabel("Score")
handles, labels = axes[0].get_legend_handles_labels()
figure.legend(handles, labels, frameon=False, fontsize=8, loc="center left", bbox_to_anchor=(0.86, 0.5))
figure.suptitle("Threshold-free ranking performance", x=0.08, ha="left", weight="bold")
figure.tight_layout(rect=(0, 0, 0.83, 1))
plt.show()

display(
    summary.rename(index=LABELS, level="pooling").round(4)
)

In [ ]:
figure, axes = plt.subplots(1, 2, figsize=(8.8, 4.2), sharey=True)
for axis, dataset, title in zip(axes, ("ragtruth_test", "publichearing_zero_shot"), ("RAGTruth test", "PublicHearingBR zero-shot")):
    values = [summary.loc[(dataset, pooling), ("Brier", "mean")] for pooling in POOLINGS]
    errors = [summary.loc[(dataset, pooling), ("Brier", "std")] for pooling in POOLINGS]
    bars = axis.bar([LABELS[pooling] for pooling in POOLINGS], values, yerr=errors, capsize=3, color=[COLORS[pooling] for pooling in POOLINGS])
    add_bar_labels(axis, bars, errors)
    axis.set_ylim(0, max(np.asarray(values) + np.asarray(errors)) + 0.060)
    axis.set_title(title, loc="left", weight="bold", pad=14)
    axis.grid(axis="y", alpha=0.2)
axes[0].set_ylabel("Brier score (lower is better)")
figure.suptitle("Score quality diagnostic", x=0.08, ha="left", weight="bold")
figure.tight_layout()
plt.show()

## Frozen threshold transfer

Each seed selects `best_f1` and `fpr10` on RAGTruth validation only. The selected threshold is then frozen and applied to RAGTruth test and PublicHearingBR. The target labels do not participate in threshold selection.

In [ ]:
target_operating = metrics.loc[
    metrics["dataset"].eq("publichearing_zero_shot") & metrics["threshold_regime"].isin(("best_f1", "fpr10"))
].copy()
operating_summary = target_operating.groupby(["threshold_regime", "pooling"])[["F1", "Recall", "FPR"]].agg(["mean", "std"])

figure, axes = plt.subplots(1, 2, figsize=(12.0, 3.8))
for axis, metric, title in zip(axes, ("F1", "FPR"), ("F1 (higher is better)", "False-positive rate (lower is better)")):
    x = np.arange(2)
    width = 0.24
    for index, pooling in enumerate(POOLINGS):
        values = [operating_summary.loc[(regime, pooling), (metric, "mean")] for regime in ("best_f1", "fpr10")]
        errors = [operating_summary.loc[(regime, pooling), (metric, "std")] for regime in ("best_f1", "fpr10")]
        bars = axis.bar(x + (index - 1) * width, values, width, yerr=errors, capsize=3, label=LABELS[pooling], color=COLORS[pooling])
        add_bar_labels(axis, bars, errors)
    axis.set_xticks(x, ("best F1", "FPR 10%"))
    axis.set_ylim(0, 0.7 if metric == "F1" else 0.13)
    axis.set_title(title, loc="left", weight="bold")
    axis.grid(axis="y", alpha=0.2)
axes[0].set_ylabel("Score")
handles, labels = axes[0].get_legend_handles_labels()
figure.legend(handles, labels, frameon=False, fontsize=8, loc="center left", bbox_to_anchor=(0.86, 0.5))
figure.suptitle("PublicHearingBR operating points from frozen RAGTruth thresholds", x=0.08, ha="left", weight="bold")
figure.tight_layout(rect=(0, 0, 0.83, 1))
plt.show()

display(operating_summary.rename(index=LABELS, level="pooling").round(4))

## Decision

Gated Attention is retained because it has the highest mean **PublicHearingBR zero-shot AUPRC**, the decision criterion used for this imbalanced target. Max has the strongest AUROC and Brier diagnostics on the target, so it remains a credible alternative; it does not, however, exceed Attention on the primary AUPRC outcome. Mean does not lead any target threshold-free metric.

This is a descriptive three-seed ablation, not a claim of statistical superiority. The grouped bootstrap artifacts remain the appropriate source for uncertainty reporting.

In [ ]:
target_ranking = (
    threshold_free.loc[threshold_free["dataset"].eq("publichearing_zero_shot")]
    .groupby("pooling")[["AUPRC", "AUROC", "Brier"]]
    .agg(["mean", "std"])
    .reindex(POOLINGS)
)
display(target_ranking.rename(index=LABELS).round(4))

primary_winner = target_ranking[("AUPRC", "mean")].idxmax()
print(f"Primary target AUPRC winner: {LABELS[primary_winner]} ({target_ranking.loc[primary_winner, ('AUPRC', 'mean')]:.4f}).")
print("Interpretation: retain Gated Attention when the decision criterion is target AUPRC; report Max alongside it for its AUROC/Brier profile.")